In [27]:
import json
import numpy as np


with open('test_data/MVl_3226_1/ball_info_3d.json', 'r') as f:
    data = json.load(f)

# Extract x_m, y_m, z_m into (N, 3) array
ball_info = np.array([[item['x'], item['y'], item['z']] for item in data])
ball_info

array([[-6.87547338e+02, -3.21997259e+02,  3.15353883e+03],
       [-7.15524148e+02, -5.84051946e+02,  4.58259046e+03],
       [-7.10980053e+02, -8.60048756e+02,  6.09959749e+03],
       [-5.98439382e+02, -9.82853569e+02,  6.69529296e+03],
       [-5.98439382e+02, -9.82853569e+02,  6.69529296e+03],
       [-6.04174193e+02, -1.30098843e+03,  8.65210057e+03],
       [-5.24450578e+02, -1.44126734e+03,  9.56120071e+03],
       [-4.15060736e+02, -1.43718367e+03,  9.58208825e+03],
       [-3.47117030e+02, -1.50812503e+03,  1.02251707e+04],
       [-3.47117030e+02, -1.50812503e+03,  1.02251707e+04],
       [-2.80047774e+02, -1.53105739e+03,  1.05460680e+04],
       [-2.27251477e+02, -1.64432676e+03,  1.16227279e+04],
       [-2.92076589e+02, -2.89880526e+03,  2.12279605e+04],
       [-1.39423054e+02, -2.13710520e+03,  1.61264930e+04],
       [-1.27785052e+02, -1.99605851e+03,  1.50269695e+04],
       [-9.12858583e+01, -2.04184261e+03,  1.58991417e+04],
       [-3.45651530e+01, -2.41817818e+03

In [28]:

dt = 1 / 30.0          # example: 120 fps (adjust to your true fps)
g  = 9.81               # m/s^2

N = ball_info.shape[0]

# State: [X, Y, Z, Vx, Vy, Vz]^T
# Initial state: position from first measurement, zero velocity
x = np.zeros((6, 1))
x[0:3, 0] = ball_info[0]

# State transition matrix F (no gravity here; gravity in control vector u)
F = np.array([
    [1, 0, 0, dt, 0,  0],
    [0, 1, 0, 0,  dt, 0],
    [0, 0, 1, 0,  0,  dt],
    [0, 0, 0, 1,  0,  0],
    [0, 0, 0, 0,  1,  0],
    [0, 0, 0, 0,  0,  1]
], dtype=float)

# Control vector u (gravity along Z)
u = np.array([
    [0.0],
    [0.0],
    [-0.5 * g * dt**2],
    [0.0],
    [0.0],
    [-g * dt]
], dtype=float)

# Process noise covariance Q (tune these)
q_pos = 1e-4
q_vel = 1e-2
Q = np.diag([q_pos, q_pos, q_pos, q_vel, q_vel, q_vel])

# Measurement matrix H (we observe X,Y,Z only)
H = np.array([
    [1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0]
], dtype=float)

# Measurement noise covariance R (tune this)
r_pos = 1e-3   # ~mm-level noise in metres
R = np.diag([r_pos, r_pos, r_pos])

# Initial state covariance P
P = np.eye(6) * 1e-2

x_filt = np.zeros((N, 6))  # store filtered states

In [29]:
for k in range(N):
    z_k = ball_info[k].reshape(3, 1)

    # 1) Predict
    x_pred = F @ x + u
    P_pred = F @ P @ F.T + Q

    # 2) Update
    y = z_k - (H @ x_pred)                          # innovation
    S = H @ P_pred @ H.T + R                        # innovation cov
    K = P_pred @ H.T @ np.linalg.inv(S)             # Kalman gain
    x = x_pred + K @ y                              # posterior state
    P = (np.eye(6) - K @ H) @ P_pred                # posterior cov

    x_filt[k, :] = x.ravel()

# Extract filtered positions
X_kalman = x_filt[:, 0]
Y_kalman = x_filt[:, 1]
Z_kalman = x_filt[:, 2]

In [30]:
import csv

# Write Kalman filtered results to CSV
with open('kalman_filtered_results.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Frame', 'X_kalman', 'Y_kalman', 'Z_kalman'])
    for i in range(N):
        writer.writerow([i, X_kalman[i], Y_kalman[i], Z_kalman[i]])

print("Results saved to kalman_filtered_results.csv")

Results saved to kalman_filtered_results.csv


In [31]:
with open("z_kalman.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Frame', 'Z_raw', "Z_kalman"])
    for z in range(N):
        writer.writerow([z, ball_info[z,2], Z_kalman[z]])

print("Z data saved to z_kalman.csv")

Z data saved to z_kalman.csv


In [26]:
ball_info[:,2]

array([ 3558.19395081,  4880.77214397,  4880.77214397,  5789.13715454,
        6584.98502973,  7833.15874645,  8089.18342755,  8089.18342755,
        9379.11491878, 10573.89027482, 11580.63880239, 12779.13500925,
       12779.13500925, 13208.31752457, 13773.67772007, 15603.20496395,
       16286.95905308, 16286.95905308, 16937.74831573, 18624.95339958,
       17511.30095468, 15092.60788809, 15092.60788809, 19133.0152572 ,
       13814.72951808, 18988.97473103, 18641.90597135, 18641.90597135,
       19891.90004754, 20943.40812806, 21207.21531055, 22806.16912854,
       22806.16912854, 41210.45585259, 31400.13358911, 31947.30643922,
       30249.80313178, 30249.80313178, 31644.48958848, 35538.19669023,
       33639.83546505, 31450.94409384])

In [23]:
Z_kalman


array([ 3558.19346031,  4230.59624975,  4499.16019242,  5000.72786971,
        5655.9547129 ,  6622.37149316,  7409.21810625,  7928.26227765,
        8789.40957362,  9843.37169217, 10937.26672245, 12128.02077046,
       12860.73330655, 13456.98668126, 14020.21945701, 15105.37780252,
       16058.55550511, 16622.4574154 , 17196.74958846, 18226.07708473,
       18371.24891448, 17351.97970249, 16588.15353543, 17758.46337513,
       16265.04020185, 17426.07453111, 18067.16674528, 18480.07819294,
       19263.67882038, 20211.89102315, 20925.91435751, 22040.60752112,
       22739.91255668, 31011.99825657, 32368.70455957, 33302.13691239,
       33002.97877416, 32612.08244382, 32795.95746166, 34473.64739989,
       34707.75806983, 33821.0380358 ])